# Qwen3 0.6b Embedding

In [ ]:
import os
import pandas as pd

In [ ]:
%%writefile constants.py
EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
MODEL_OUTPUT_PATH = '/kaggle/input/qwen3-8b-embedding'
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"

CLEAN_TEXT = True
TOP_K = 500
BATCH_SIZE = 128

In [ ]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist

from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm

from constants import CLEAN_TEXT


def build_prompt(row):
    return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""


def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )



def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv").sample(frac=0.6, random_state=42).reset_index(drop=True)

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)    
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    
    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: -1,
            }
        )

    return dataframe

In [ ]:
%%writefile semantic.py
import pandas as pd
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
from peft import PeftModel, PeftConfig


from utils import get_dataframe_to_train, prepare_dataframe
from constants import DATA_PATH, EMBDEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE, MODEL_OUTPUT_PATH



def get_scores(test_dataframe):
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)
    
    # Load base model
    model = AutoModelForCausalLM.from_pretrained(EMBDEDDING_MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(EMBDEDDING_MODEL_PATH)
    
    # Load adapter configuration and model
    adapter_config = PeftConfig.from_pretrained(MODEL_OUTPUT_PATH)
    lora_model = PeftModel.from_pretrained(model, MODEL_OUTPUT_PATH, config=adapter_config)
    merged_model = lora_model.merge_and_unload()
    tokenizer.save_pretrained("Qwen3Emb_Finetuned")
    merged_model.save_pretrained("Qwen3Emb_Finetuned")

    # 4. Tạo lại SentenceTransformer từ encoder đã merge
    embedding_model = SentenceTransformer(model_name_or_path="Qwen3Emb_Finetuned", device="cuda")

    print('Done loading model!')

    result = []
    for rule in tqdm(test_dataframe["rule"].unique(), desc=f"Generate scores for each rule"):
        test_dataframe_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe_part.reset_index(names="row_id")
        
        query_embeddings = embedding_model.encode(
            sentences=test_dataframe_part["prompt"].tolist(),
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        document_embeddings = embedding_model.encode(
            sentences=corpus_dataframe_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        test_dataframe_part["semantic"] = semantic_search(
            query_embeddings,
            document_embeddings,
            top_k=TOP_K,
            score_function=dot_score,
        )
        def get_score(semantic):
            semantic = pd.DataFrame(semantic)
            semantic = semantic.merge(
                corpus_dataframe_part[["row_id", "rule_violation"]],
                how="left",
                left_on="corpus_id",
                right_on="row_id",
            )
            semantic["score"] = semantic["score"]*semantic["rule_violation"]
            return semantic["score"].sum()
            
        tqdm.pandas(desc=f"Add label for {rule=}")
        test_dataframe_part["rule_violation"] = test_dataframe_part["semantic"].progress_apply(get_score)
        result.append(test_dataframe_part[["row_id", "rule_violation"]].copy())
        
    submission = pd.concat(result, axis=0)
    return submission


def generate_submission():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)
    
    submission = get_scores(test_dataframe)
    submission = test_dataframe[["row_id"]].merge(submission, on="row_id", how="left")
    submission.to_csv("submission_qwen3.csv", index=False)


if __name__ == "__main__":
    generate_submission()

In [ ]:
# !python semantic.py

In [ ]:
#### test-train

In [11]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/qwen2.5/transformers/0.5b-instruct-gptq-int4/1"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

# Embedding model for semantic search
EMBEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"
TOP_K_SEMANTIC = 100
EMBEDDING_BATCH_SIZE = 128

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

Overwriting constants.py


In [12]:
%%writefile utils.py
import pandas as pd
import torch
import numpy as np
from datasets import Dataset
# from sentence_transformers import SentenceTransformer
# from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
from constants import (
    POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT,
    EMBEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY, TOP_K_SEMANTIC, EMBEDDING_BATCH_SIZE
)
import random
random.seed(42)
np.random.seed(42)


def build_prompt(row):
    return f"""
{BASE_PROMPT}

Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
{COMPLETE_PHRASE} Yes

2) {row["negative_example"]}
{COMPLETE_PHRASE} No

---
Comment: {row["body"]}
{COMPLETE_PHRASE}"""



def get_dataframe_to_train(data_path):
    """Load pre-computed semantic training data"""
    dataframe = pd.read_csv("semantic_training_data.csv")
    return dataframe


def build_dataset(dataframe):
    """Build dataset for training"""
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]
    if "rule_violation" in dataframe:
        dataframe["completion"] = dataframe["rule_violation"].map(
            {
                1: POSITIVE_ANSWER,
                0: NEGATIVE_ANSWER,
            }
        )
        columns.append("completion")

    dataframe = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe)
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)
    return dataset

Overwriting utils.py


In [13]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH, LORA_PATH


def main():
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        num_train_epochs=1,

        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,

        optim="paged_adamw_8bit",
        learning_rate=1e-4, #keep high, lora usually likes high.
        weight_decay=0.01,
        max_grad_norm=1.0,

        lr_scheduler_type="cosine",
        warmup_ratio=0.03,

        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),
        dataloader_pin_memory=True,

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        save_strategy="no",
        report_to="none",

        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        BASE_MODEL_PATH,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )

    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

Overwriting train.py


In [14]:
%%writefile inference.py
import os
os.environ["VLLM_USE_V1"] = "0"

import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from sentence_transformers import SentenceTransformer
from utils import build_dataset, build_labeled_corpus, get_semantic_examples
from constants import (
    BASE_MODEL_PATH, LORA_PATH, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER,
    EMBEDDING_MODEL_PATH
)
import multiprocessing as mp


def prepare_test_data_with_semantic_examples(df_slice):
    """Prepare test data with semantically selected examples"""
    # Build corpus for semantic search
    corpus_df = build_labeled_corpus(DATA_PATH)

    # Load embedding model
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_PATH, device="cuda" if torch.cuda.is_available() else "cpu")

    results = []
    for _, row in df_slice.iterrows():
        pos_example, neg_example = get_semantic_examples(
            row["body"], row["rule"], row["subreddit"], corpus_df, embedding_model
        )

        row_dict = row.to_dict()
        row_dict["positive_example"] = pos_example
        row_dict["negative_example"] = neg_example
        results.append(row_dict)

    return pd.DataFrame(results)


def run_inference_on_device(df_slice):
    """Run vLLM inference on current process visible GPU"""
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        quantization="gptq",
        tensor_parallel_size=1,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2836,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )

    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])

    # Prepare data with semantic examples
    df_with_examples = prepare_test_data_with_semantic_examples(df_slice)
    test_dataset = build_dataset(df_with_examples)
    texts = test_dataset["prompt"]

    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )

    log_probs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    predictions = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    predictions["row_id"] = df_slice["row_id"].values
    return predictions


def worker(device_id, df_slice, return_dict):
    # Limit process to single GPU
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] Running on GPU {device_id}, data size={len(df_slice)}")

    preds = run_inference_on_device(df_slice)
    return_dict[device_id] = preds


def main():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")

    # Split data for parallel processing
    mid = len(test_dataframe) // 2
    df0 = test_dataframe.iloc[:mid].reset_index(drop=True)
    df1 = test_dataframe.iloc[mid:].reset_index(drop=True)

    manager = mp.Manager()
    return_dict = manager.dict()

    # Run parallel inference
    p0 = mp.Process(target=worker, args=(0, df0, return_dict))
    p1 = mp.Process(target=worker, args=(1, df1, return_dict))
    p0.start()
    p1.start()
    p0.join()
    p1.join()

    # Combine results
    predictions = pd.concat([return_dict[0], return_dict[1]], ignore_index=True)

    # Build submission
    submission = predictions[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    rq = submission['rule_violation'].rank(method='average') / (len(submission) + 1)
    submission['rule_violation'] = rq

    submission.to_csv("submission.csv", index=False)
    # print("✅ Saved submission_semantic.csv")


if __name__ == "__main__":
    main()

Overwriting inference.py


In [15]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_batch_size: 64
  train_micro_batch_size_per_gpu: 4

  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false

  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5

  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false

  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1

distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

Overwriting accelerate_config.yaml


In [ ]:
%%writefile prepare_semantic_data.py
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
from constants import (
  DATA_PATH, EMBEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY,
  TOP_K_SEMANTIC, EMBEDDING_BATCH_SIZE
)
import numpy as np
import random
random.seed(42)
np.random.seed(42)


def build_labeled_corpus(data_path):
  """Build comprehensive labeled corpus from all available data"""
  train_dataset = pd.read_csv(f"{data_path}/train.csv")
  test_dataset = pd.read_csv(f"{data_path}/test.csv")

  corpus = []

  # Add train data
  for _, row in train_dataset.iterrows():
      corpus.append({
          "body": row["body"],
          "rule": row["rule"],
          "subreddit": row["subreddit"],
          "rule_violation": row["rule_violation"]
      })

  # Add positive examples from test data
  for _, row in test_dataset.iterrows():
      for i in [1, 2]:
          corpus.append({
              "body": row[f"positive_example_{i}"],
              "rule": row["rule"],
              "subreddit": row["subreddit"],
              "rule_violation": 1
          })

  # Add negative examples from test data
  for _, row in test_dataset.iterrows():
      for i in [1, 2]:
          corpus.append({
              "body": row[f"negative_example_{i}"],
              "rule": row["rule"],
              "subreddit": row["subreddit"],
              "rule_violation": 0
          })

  corpus_df = pd.DataFrame(corpus).drop_duplicates().reset_index(drop=True)
  corpus_df["corpus_id"] = corpus_df.index
  return corpus_df


# def get_semantic_examples(target_comment, target_rule, target_subreddit, corpus_df, embedding_model):
#   """Find semantically similar positive and negative examples, avoiding exact matches"""

#   # Filter corpus by rule and subreddit
#   rule_corpus = corpus_df[
#       (corpus_df["rule"] == target_rule) &
#       (corpus_df["subreddit"] == target_subreddit) &
#       (corpus_df["body"] != target_comment)  # Avoid exact matches for leakage prevention
#   ].reset_index(drop=True)

#   if len(rule_corpus) == 0:
#       # Fallback to any rule if no matches in same rule/subreddit
#       rule_corpus = corpus_df[corpus_df["body"] != target_comment].reset_index(drop=True)

#   # Encode target comment
#   target_embedding = embedding_model.encode(
#       [target_comment],
#       prompt=EMBEDDING_MODEL_QUERY,
#       batch_size=1,
#       convert_to_tensor=True,
#       normalize_embeddings=True,
#   )

#   # Encode corpus
#   corpus_embeddings = embedding_model.encode(
#       rule_corpus["body"].tolist(),
#       batch_size=EMBEDDING_BATCH_SIZE,
#       convert_to_tensor=True,
#       normalize_embeddings=True,
#       show_progress_bar=False,
#   )

#   # Find most similar examples
#   search_results = semantic_search(
#       target_embedding,
#       corpus_embeddings,
#       top_k=min(TOP_K_SEMANTIC, len(rule_corpus)),
#       score_function=dot_score,
#   )[0]

#   # Get positive and negative examples separately
#   positive_examples = []
#   negative_examples = []

#   for result in search_results:
#       corpus_idx = result["corpus_id"]
#       example_row = rule_corpus.iloc[corpus_idx]

#       if example_row["rule_violation"] == 1 and len(positive_examples) == 0:
#           positive_examples.append(example_row["body"])
#       elif example_row["rule_violation"] == 0 and len(negative_examples) == 0:
#           negative_examples.append(example_row["body"])

#       # Stop when we have both types
#       if len(positive_examples) > 0 and len(negative_examples) > 0:
#           break

#   # Fallback to random if semantic search fails
#   if len(positive_examples) == 0:
#       pos_candidates = rule_corpus[rule_corpus["rule_violation"] == 1]
#       if len(pos_candidates) > 0:
#           positive_examples.append(pos_candidates.sample(1)["body"].iloc[0])

#   if len(negative_examples) == 0:
#       neg_candidates = rule_corpus[rule_corpus["rule_violation"] == 0]
#       if len(neg_candidates) > 0:
#           negative_examples.append(neg_candidates.sample(1)["body"].iloc[0])

#   return (
#       positive_examples[0] if positive_examples else "No positive example found",
#       negative_examples[0] if negative_examples else "No negative example found"
#   )


def main():
  # Build corpus for semantic search
  corpus_df = build_labeled_corpus(DATA_PATH)

  # Load embedding model
  print("Loading embedding model...")
  embedding_model = SentenceTransformer(EMBEDDING_MODEL_PATH, device="cuda")

  train_dataset = pd.read_csv(f"{DATA_PATH}/train.csv")
  test_dataset = pd.read_csv(f"{DATA_PATH}/test.csv")#.sample(frac=0.5, random_state=42).reset_index(drop=True)

  all_data = []

  # Get unique rule/subreddit combinations to optimize corpus encoding
  all_targets = []

  # Collect all targets to process
  for _, row in train_dataset.iterrows():
      all_targets.append({
          "body": row["body"],
          "rule": row["rule"],
          "subreddit": row["subreddit"],
          "rule_violation": row["rule_violation"],
          "source": "train"
      })

  for violation_type in ["positive", "negative"]:
      for i in range(1, 3):
          for _, row in test_dataset.iterrows():
              all_targets.append({
                  "body": row[f"{violation_type}_example_{i}"],
                  "rule": row["rule"],
                  "subreddit": row["subreddit"],
                  "rule_violation": 1 if violation_type == "positive" else 0,
                  "source": f"{violation_type}_{i}"
              })

  # Group by rule/subreddit for efficient processing
  rule_groups = {}
  for target in all_targets:
      key = (target["rule"], target["subreddit"])
      if key not in rule_groups:
          rule_groups[key] = []
      rule_groups[key].append(target)

  print(f"Processing {len(all_targets)} targets across {len(rule_groups)} rule/subreddit combinations")

  # Process each rule/subreddit group
  for (rule, subreddit), targets in tqdm(rule_groups.items(), desc="Processing rule groups"):
      # Filter and encode corpus once per group
      rule_corpus = corpus_df[
          (corpus_df["rule"] == rule) &
          (corpus_df["subreddit"] == subreddit)
      ].reset_index(drop=True)

      if len(rule_corpus) == 0:
          # Fallback to any rule corpus
          rule_corpus = corpus_df.reset_index(drop=True)

      # Pre-encode corpus for this group
      print(f"Encoding corpus for {rule[:30]}... ({len(rule_corpus)} examples)")
      corpus_embeddings = embedding_model.encode(
          rule_corpus["body"].tolist(),
          batch_size=EMBEDDING_BATCH_SIZE,
          convert_to_tensor=True,
          normalize_embeddings=True,
          show_progress_bar=False,
      )

      # Extract target bodies and encode in batch
      target_bodies = [t["body"] for t in targets]

      print(f"Encoding {len(target_bodies)} target comments...")
      target_embeddings = embedding_model.encode(
          target_bodies,
          prompt=EMBEDDING_MODEL_QUERY,
          batch_size=EMBEDDING_BATCH_SIZE,
          convert_to_tensor=True,
          normalize_embeddings=True,
          show_progress_bar=False,
      )

      # Perform batch semantic search
      search_results_batch = semantic_search(
          target_embeddings,
          corpus_embeddings,
          top_k=min(TOP_K_SEMANTIC, len(rule_corpus)),
          score_function=dot_score,
      )

      # Process results for each target
      for i, target in enumerate(targets):
          search_results = search_results_batch[i]

          # Filter out exact matches
          filtered_results = []
          for result in search_results:
              corpus_idx = result["corpus_id"]
              if rule_corpus.iloc[corpus_idx]["body"] != target["body"]:
                  filtered_results.append(result)

          # Get positive and negative examples
          positive_examples = []
          negative_examples = []

          for result in filtered_results:
              corpus_idx = result["corpus_id"]
              example_row = rule_corpus.iloc[corpus_idx]

              if example_row["rule_violation"] == 1 and len(positive_examples) == 0:
                  positive_examples.append(example_row["body"])
              elif example_row["rule_violation"] == 0 and len(negative_examples) == 0:
                  negative_examples.append(example_row["body"])

              if len(positive_examples) > 0 and len(negative_examples) > 0:
                  break

          # Fallback to random if semantic search fails
          if len(positive_examples) == 0:
              pos_candidates = rule_corpus[rule_corpus["rule_violation"] == 1]
              if len(pos_candidates) > 0:
                  positive_examples.append(pos_candidates.sample(1)["body"].iloc[0])

          if len(negative_examples) == 0:
              neg_candidates = rule_corpus[rule_corpus["rule_violation"] == 0]
              if len(neg_candidates) > 0:
                  negative_examples.append(neg_candidates.sample(1)["body"].iloc[0])

          # Add to results
          all_data.append({
              "body": target["body"],
              "rule": target["rule"],
              "subreddit": target["subreddit"],
              "rule_violation": target["rule_violation"],
              "positive_example": positive_examples[0] if positive_examples else "No positive example found",
              "negative_example": negative_examples[0] if negative_examples else "No negative example found"
          })

      # Clear embeddings to save memory
      del corpus_embeddings, target_embeddings
      torch.cuda.empty_cache()

  # Save to CSV
  result_df = pd.DataFrame(all_data).drop_duplicates().reset_index(drop=True)
  result_df.to_csv("semantic_training_data.csv", index=False)
  print(f"✅ Saved semantic_training_data.csv with {len(result_df)} examples")

  # Clear GPU memory
  del embedding_model
  torch.cuda.empty_cache()


if __name__ == "__main__":
  main()

In [ ]:
!python prepare_semantic_data.py

In [16]:
!accelerate launch --config_file accelerate_config.yaml train.py

Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/accelerate/commands/accelerate_cli.py", line 48, in main
    args.func(args)
  File "/usr/local/lib/python3.11/dist-packages/accelerate/commands/launch.py", line 1177, in launch_command
    deepspeed_launcher(args)
  File "/usr/local/lib/python3.11/dist-packages/accelerate/commands/launch.py", line 824, in deepspeed_launcher
    raise ImportError("DeepSpeed is not installed => run `pip3 install deepspeed` or build it from source.")
ImportError: DeepSpeed is not installed => run `pip3 install deepspeed` or build it from source.


In [17]:
!python inference.py

Traceback (most recent call last):
  File "/kaggle/working/inference.py", line 4, in <module>
    import vllm
ModuleNotFoundError: No module named 'vllm'


In [18]:
!head submission.csv

head: cannot open 'submission_qwen.csv' for reading: No such file or directory
